# Route Resilience — Phase 2: Deep Learning Road Extraction Pipeline

This notebook orchestrates the deep learning training pipeline for **Phase 2: Occlusion-Robust Road Extraction**.

### Project Context
- **Goal**: Train a semantic segmentation model that extracts road centerlines and is robust to shadows, trees, vehicles, and clouds.
- **Modularity**: All core reusable logic (loaders, models, losses, training loop) is implemented in clean Python modules under `src/segmentation/` so they can be reused in inference, APIs, and dashboard systems in Phase 3 & 4.
- **Kaggle Execution**: The setup cell below automatically recreates the modular folder structure and writes the source files when running in a clean Kaggle kernel.

## Step 1: Write Modular Python Code to disk (Kaggle Compatibility)

In [ ]:
import os

# Create directories if they do not exist
os.makedirs("src/segmentation/datasets", exist_ok=True)
os.makedirs("src/segmentation/models", exist_ok=True)
os.makedirs("src/segmentation/losses", exist_ok=True)
os.makedirs("src/segmentation/evaluation", exist_ok=True)
os.makedirs("src/segmentation/training", exist_ok=True)
os.makedirs("src/segmentation/inference", exist_ok=True)
os.makedirs("configs", exist_ok=True)

print("✓ Modular directories initialized.")

In [ ]:
%%writefile src/segmentation/datasets/road_dataset.py
import os
import random
from typing import Any, Dict, List, Optional, Tuple
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

try:
    import noise
    _HAS_NOISE = True
except ImportError:
    _HAS_NOISE = False

DEFAULT_OCCLUSION_CFG = {
    "tree_canopy": {
        "light": {"coverage_min": 0.10, "coverage_max": 0.25},
        "medium": {"coverage_min": 0.25, "coverage_max": 0.50},
        "heavy": {"coverage_min": 0.50, "coverage_max": 0.75},
        "noise_scale": 0.05,
        "blob_size_range": [30, 120],
        "color_variation": 20
    },
    "building_shadow": {
        "light": {"coverage_min": 0.05, "coverage_max": 0.20},
        "medium": {"coverage_min": 0.20, "coverage_max": 0.40},
        "heavy": {"coverage_min": 0.40, "coverage_max": 0.65},
        "sun_angle_range": [30, 150],
        "shadow_length_range": [0.5, 2.0],
        "darkening_factor": 0.35
    },
    "vehicle": {
        "light": {"count_range": [1, 3]},
        "medium": {"count_range": [3, 8]},
        "heavy": {"count_range": [8, 20]},
        "size_range_w": [8, 25],
        "size_range_h": [15, 45]
    },
    "cloud_cover": {
        "light": {"coverage_min": 0.05, "coverage_max": 0.20, "opacity_range": [0.2, 0.5]},
        "medium": {"coverage_min": 0.20, "coverage_max": 0.45, "opacity_range": [0.5, 0.75]},
        "heavy": {"coverage_min": 0.45, "coverage_max": 0.70, "opacity_range": [0.75, 0.95]},
        "blur_sigma_range": [20, 60]
    }
}

def _gaussian_blob(h: int, w: int, cx: int, cy: int, radius: int) -> np.ndarray:
    Y, X = np.ogrid[:h, :w]
    dist2 = (X - cx) ** 2 + (Y - cy) ** 2
    sigma = max(1.0, radius / 2.0)
    blob = np.exp(-dist2 / (2 * sigma ** 2))
    return blob.astype(np.float32)

def _perlin_blob(h: int, w: int, cx: int, cy: int, radius: int, scale: float = 0.05) -> np.ndarray:
    blob = np.zeros((h, w), dtype=np.float32)
    offset_x = random.random() * 1000
    offset_y = random.random() * 1000
    for y in range(max(0, cy - radius), min(h, cy + radius)):
        for x in range(max(0, cx - radius), min(w, cx + radius)):
            dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
            if dist < radius:
                n = noise.pnoise2(x * scale + offset_x, y * scale + offset_y, octaves=4)
                falloff = 1.0 - (dist / radius) ** 2
                blob[y, x] = max(0.0, (n + 0.5) * falloff)
    return blob

def _compute_shadow_polygon(ox: int, oy: int, bld_w: int, bld_h: int, sun_angle_deg: float, shadow_len: float) -> List[Tuple[int, int]]:
    angle_rad = np.radians(sun_angle_deg)
    dx = int(shadow_len * np.cos(angle_rad))
    dy = int(shadow_len * np.sin(angle_rad))
    corners = [(ox, oy), (ox + bld_w, oy), (ox + bld_w, oy + bld_h), (ox, oy + bld_h)]
    shadow = [(x + dx, y + dy) for x, y in corners]
    return corners + shadow[::-1]

def apply_tree_canopy(img: np.ndarray, mask: np.ndarray, severity: str, cfg: Dict) -> np.ndarray:
    h, w = img.shape[:2]
    sev_cfg = cfg.get(severity, {})
    cov_min = sev_cfg.get("coverage_min", 0.25)
    cov_max = sev_cfg.get("coverage_max", 0.50)
    blob_min, blob_max = cfg.get("blob_size_range", [30, 120])
    color_var = cfg.get("color_variation", 20)
    target_coverage = random.uniform(cov_min, cov_max)
    occluded = img.copy().astype(np.float32)
    occlusion_mask = np.zeros((h, w), dtype=np.uint8)
    total_px = h * w
    covered = 0
    attempts = 0
    while covered / total_px < target_coverage and attempts < 100:
        attempts += 1
        blob_r = random.randint(blob_min // 2, blob_max // 2)
        cx = random.randint(blob_r, w - blob_r)
        cy = random.randint(blob_r, h - blob_r)
        if _HAS_NOISE:
            blob = _perlin_blob(h, w, cx, cy, blob_r, scale=cfg.get("noise_scale", 0.05))
        else:
            blob = _gaussian_blob(h, w, cx, cy, blob_r)
        blob = (blob > 0.35).astype(np.uint8)
        base_green = np.array([34, 85, 34], dtype=np.float32)
        color = base_green + np.random.uniform(-color_var, color_var, 3)
        color = np.clip(color, 0, 255)
        alpha = np.where(blob, random.uniform(0.55, 0.80), 0.0)
        alpha = np.expand_dims(alpha, axis=-1)
        occluded = occluded * (1 - alpha) + color * alpha
        occlusion_mask = np.maximum(occlusion_mask, (blob * 255).astype(np.uint8))
        covered = int(np.sum(occlusion_mask > 0))
    return np.clip(occluded, 0, 255).astype(np.uint8)

def apply_building_shadow(img: np.ndarray, mask: np.ndarray, severity: str, cfg: Dict) -> np.ndarray:
    h, w = img.shape[:2]
    sev_cfg = cfg.get(severity, {})
    cov_min = sev_cfg.get("coverage_min", 0.20)
    cov_max = sev_cfg.get("coverage_max", 0.40)
    darkening = cfg.get("darkening_factor", 0.35)
    sun_min, sun_max = cfg.get("sun_angle_range", [30, 150])
    len_min, len_max = cfg.get("shadow_length_range", [0.5, 2.0])
    target_coverage = random.uniform(cov_min, cov_max)
    occluded = img.copy().astype(np.float32)
    occlusion_mask = np.zeros((h, w), dtype=np.uint8)
    total_px = h * w
    covered = 0
    for _ in range(50):
        if covered / total_px >= target_coverage:
            break
        ox = random.randint(w // 8, 7 * w // 8)
        oy = random.randint(h // 8, 7 * h // 8)
        bld_w = random.randint(20, 80)
        bld_h = random.randint(20, 80)
        sun_angle = random.uniform(sun_min, sun_max)
        shadow_len = random.uniform(len_min, len_max) * max(bld_w, bld_h)
        shadow_poly = _compute_shadow_polygon(ox, oy, bld_w, bld_h, sun_angle, shadow_len)
        shadow_poly = np.array(shadow_poly, dtype=np.int32)
        shadow_img = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(shadow_img, [shadow_poly], 255)
        dist_map = cv2.distanceTransform(shadow_img, cv2.DIST_L2, 5)
        if dist_map.max() > 0:
            gradient = 1.0 - (dist_map / dist_map.max()) * 0.4
        else:
            gradient = np.ones((h, w), dtype=np.float32)
        alpha = np.where(shadow_img > 0, gradient * (1.0 - darkening), 0.0)
        alpha = np.expand_dims(alpha, axis=-1)
        occluded = occluded * (1.0 - alpha * (1.0 - darkening))
        occlusion_mask = np.maximum(occlusion_mask, shadow_img)
        covered = int(np.sum(occlusion_mask > 0))
    return np.clip(occluded, 0, 255).astype(np.uint8)

def apply_vehicle(img: np.ndarray, mask: np.ndarray, severity: str, cfg: Dict) -> np.ndarray:
    h, w = img.shape[:2]
    sev_cfg = cfg.get(severity, {})
    count_range = sev_cfg.get("count_range", [3, 8])
    veh_w_range = cfg.get("size_range_w", [8, 25])
    veh_h_range = cfg.get("size_range_h", [15, 45])
    n_vehicles = random.randint(*count_range)
    occluded = img.copy()
    road_ys, road_xs = np.where(mask > 0)
    if len(road_ys) == 0:
        road_ys = np.arange(0, h)
        road_xs = np.arange(0, w)
    vehicle_colors = [(200,200,200), (40,40,40), (180,30,30), (30,30,160), (200,180,30), (50,100,50)]
    for _ in range(n_vehicles):
        idx = random.randint(0, len(road_ys) - 1)
        cx, cy = int(road_xs[idx]), int(road_ys[idx])
        vw = random.randint(*veh_w_range)
        vh = random.randint(*veh_h_range)
        color = random.choice(vehicle_colors)
        x1, y1 = max(0, cx - vw // 2), max(0, cy - vh // 2)
        x2, y2 = min(w, cx + vw // 2), min(h, cy + vh // 2)
        cv2.rectangle(occluded, (x1, y1), (x2, y2), color, -1)
        cv2.rectangle(occluded, (x1 + 1, y1 + 1), (x2 - 1, y1 + max(2, vh // 6)), tuple(min(255, c + 60) for c in color), -1)
    return occluded

def apply_cloud_cover(img: np.ndarray, severity: str, cfg: Dict) -> np.ndarray:
    h, w = img.shape[:2]
    sev_cfg = cfg.get(severity, {})
    cov_min = sev_cfg.get("coverage_min", 0.20)
    cov_max = sev_cfg.get("coverage_max", 0.45)
    opacity_min, opacity_max = sev_cfg.get("opacity_range", [0.5, 0.75])
    sigma_min, sigma_max = cfg.get("blur_sigma_range", [20, 60])
    target_coverage = random.uniform(cov_min, cov_max)
    occluded = img.copy().astype(np.float32)
    cloud_layer = np.zeros((h, w), dtype=np.float32)
    total_px = h * w
    covered = 0
    for _ in range(30):
        if covered / total_px >= target_coverage:
            break
        sigma = random.uniform(sigma_min, sigma_max)
        cx, cy = random.randint(0, w), random.randint(0, h)
        blob = _gaussian_blob(h, w, cx, cy, radius=int(sigma * 1.5))
        cloud_layer = np.maximum(cloud_layer, blob)
        covered = int(np.sum(cloud_layer > 0.1))
    cloud_blurred = cv2.GaussianBlur(cloud_layer, (0, 0), sigmaX=15)
    cloud_blurred = np.clip(cloud_blurred / (cloud_blurred.max() + 1e-8), 0, 1)
    opacity = random.uniform(opacity_min, opacity_max)
    cloud_color = np.array([240, 242, 245], dtype=np.float32)
    alpha_map = np.expand_dims(cloud_blurred * opacity, axis=-1)
    occluded = occluded * (1.0 - alpha_map) + cloud_color * alpha_map
    return np.clip(occluded, 0, 255).astype(np.uint8)

class SyntheticOcclusionTransform:
    def __init__(self, p: float = 0.5, types: Optional[List[str]] = None, config: Optional[Dict] = None):
        self.p = p
        self.types = types or ["tree_canopy", "building_shadow", "vehicle", "cloud_cover"]
        self.config = config or DEFAULT_OCCLUSION_CFG
    def __call__(self, img: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if random.random() > self.p:
            return img, mask
        occ_type = random.choice(self.types)
        severity = random.choice(["light", "medium", "heavy"])
        if occ_type == "tree_canopy":
            img = apply_tree_canopy(img, mask, severity, self.config["tree_canopy"])
        elif occ_type == "building_shadow":
            img = apply_building_shadow(img, mask, severity, self.config["building_shadow"])
        elif occ_type == "vehicle":
            img = apply_vehicle(img, mask, severity, self.config["vehicle"])
        elif occ_type == "cloud_cover":
            img = apply_cloud_cover(img, severity, self.config["cloud_cover"])
        return img, mask

def resolve_tile_path(csv_path_val: str, dataset_dir: str) -> str:
    if not csv_path_val: return ""
    basename = os.path.basename(csv_path_val)
    is_mask = "masks" in csv_path_val or "mask" in basename.lower()
    subdir = "tiles/train/masks" if is_mask else "tiles/train/images"
    candidate1 = os.path.join(dataset_dir, subdir, basename)
    if os.path.exists(candidate1):
        return candidate1
    norm_path = os.path.normpath(csv_path_val)
    parts = norm_path.split(os.sep)
    if "data" in parts:
        data_idx = parts.index("data")
        candidate2 = os.path.join(dataset_dir, *parts[data_idx + 1:])
        if os.path.exists(candidate2): return candidate2
    if os.path.exists(csv_path_val): return csv_path_val
    return candidate1

class RoadDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_dir: str, transform: Optional[Any] = None, occlusion_transform: Optional[SyntheticOcclusionTransform] = None):
        self.df = df.reset_index(drop=True)
        self.dataset_dir = dataset_dir
        self.transform = transform
        self.occlusion_transform = occlusion_transform
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        img_path = resolve_tile_path(row["image_tile_path"], self.dataset_dir)
        msk_path = resolve_tile_path(row["mask_tile_path"], self.dataset_dir)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)
        if self.occlusion_transform is not None:
            image, mask = self.occlusion_transform(image, mask)
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]
        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image.transpose(2, 0, 1))
        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask)
        if len(mask.shape) == 2:
            mask = mask.unsqueeze(0)
        mask = (mask > 0.5).float()
        return image, mask

In [ ]:
%%writefile src/segmentation/models/model_factory.py
import logging
from typing import Any, Callable, Dict
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

logger = logging.getLogger(__name__)

class ModelFactory:
    _registry: Dict[str, Callable[..., nn.Module]] = {}
    @classmethod
    def register(cls, name: str) -> Callable:
        def decorator(builder_fn: Callable[..., nn.Module]) -> Callable:
            cls._registry[name.lower().strip()] = builder_fn
            return builder_fn
        return decorator
    @classmethod
    def create(cls, name: str, **kwargs) -> nn.Module:
        key = name.lower().strip()
        if key not in cls._registry:
            raise KeyError(f"Model '{name}' not registered. Registered: {list(cls._registry.keys())}")
        return cls._registry[key](**kwargs)

@ModelFactory.register("deeplabv3plus")
def build_deeplabv3plus(encoder_name: str = "resnet34", encoder_weights: str = "imagenet", classes: int = 1, **kwargs) -> nn.Module:
    return smp.DeepLabV3Plus(encoder_name=encoder_name, encoder_weights=encoder_weights, classes=classes, **kwargs)

@ModelFactory.register("unet")
def build_unet(encoder_name: str = "resnet34", encoder_weights: str = "imagenet", classes: int = 1, **kwargs) -> nn.Module:
    return smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights, classes=classes, **kwargs)

@ModelFactory.register("unetplusplus")
def build_unetplusplus(encoder_name: str = "resnet34", encoder_weights: str = "imagenet", classes: int = 1, **kwargs) -> nn.Module:
    return smp.UnetPlusPlus(encoder_name=encoder_name, encoder_weights=encoder_weights, classes=classes, **kwargs)

@ModelFactory.register("segformer")
def build_segformer(encoder_name: str = "nvidia/mit-b0", classes: int = 1, **kwargs) -> nn.Module:
    try:
        from transformers import SegformerForSemanticSegmentation
    except ImportError:
        raise ImportError("transformers package is required for SegFormer. Run pip install transformers")
    class SegFormerWrapper(nn.Module):
        def __init__(self, model_name: str, num_labels: int):
            super().__init__()
            self.model = SegformerForSemanticSegmentation.from_pretrained(model_name, num_labels=num_labels, ignore_mismatched_sizes=True)
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            outputs = self.model(pixel_values=x)
            logits = outputs.logits
            upsampled = nn.functional.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
            return upsampled
    return SegFormerWrapper(encoder_name, classes)

In [ ]:
%%writefile src/segmentation/losses/losses.py
import torch
import torch.nn as nn

class DiceLoss(nn.Module):
    def __init__(self, eps: float = 1e-7):
        super().__init__()
        self.eps = eps
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        probs, targets = probs.view(-1), targets.view(-1)
        intersection = (probs * targets).sum()
        union = probs.sum() + targets.sum()
        dice = (2.0 * intersection + self.eps) / (union + self.eps)
        return 1.0 - dice

class JaccardLoss(nn.Module):
    def __init__(self, eps: float = 1e-7):
        super().__init__()
        self.eps = eps
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        probs, targets = probs.view(-1), targets.view(-1)
        intersection = (probs * targets).sum()
        union = probs.sum() + targets.sum() - intersection
        jaccard = (intersection + self.eps) / (union + self.eps)
        return 1.0 - jaccard

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5, jaccard_weight: float = 0.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.jaccard = JaccardLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.jaccard_weight = jaccard_weight
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss = 0.0
        if self.bce_weight > 0: loss += self.bce_weight * self.bce(logits, targets)
        if self.dice_weight > 0: loss += self.dice_weight * self.dice(logits, targets)
        if self.jaccard_weight > 0: loss += self.jaccard_weight * self.jaccard(logits, targets)
        return loss

In [ ]:
%%writefile src/segmentation/evaluation/metrics.py
from typing import Dict
import torch

def compute_batch_metrics(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5) -> Dict[str, float]:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds, targets = preds.view(-1), targets.view(-1)
    tp = (preds * targets).sum()
    fp = (preds * (1.0 - targets)).sum()
    fn = ((1.0 - preds) * targets).sum()
    eps = 1e-7
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    return {
        "precision": precision.item(),
        "recall": recall.item(),
        "f1_score": f1.item(),
        "dice": f1.item(),
        "iou": iou.item()
    }

In [ ]:
%%writefile src/segmentation/training/trainer.py
import os
import time
import yaml
import logging
from typing import Any, Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from src.segmentation.evaluation.metrics import compute_batch_metrics

logger = logging.getLogger(__name__)

class Trainer:
    def __init__(self, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, val_dataset: Dataset, criterion: nn.Module, optimizer: optim.Optimizer, scheduler: Optional[Any] = None, config: Optional[Dict[str, Any]] = None):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.val_dataset = val_dataset
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.config = config or {}
        cfg_train = self.config.get("training", {})
        self.epochs = cfg_train.get("epochs", 15)
        self.grad_clip_norm = cfg_train.get("grad_clip_norm", 1.0)
        self.early_stopping_patience = cfg_train.get("early_stopping_patience", 5)
        self.device = torch.device(cfg_train.get("device", "cuda") if torch.cuda.is_available() else "cpu")
        self.accumulation_steps = cfg_train.get("accumulation_steps", 1)
        cfg_paths = self.config.get("paths", {})
        self.output_dir = cfg_paths.get("output_dir", "/kaggle/working/output")
        self.is_master = True
        self._setup_experiment_dir()
        self.start_epoch = 1
        self.best_metric = 0.0
        self.epochs_no_improve = 0
        self.history = { "epoch": [], "train_loss": [], "val_loss": [], "val_dice": [], "val_iou": [], "val_precision": [], "val_recall": [], "learning_rate": [] }
        self.scaler = GradScaler(enabled=(self.device.type == "cuda"))
        self.num_val_samples = min(3, len(val_dataset))
        self.model.to(self.device)

    def _setup_experiment_dir(self):
        os.makedirs(self.output_dir, exist_ok=True)
        exps = [d for d in os.listdir(self.output_dir) if d.startswith("experiment_") and os.path.isdir(os.path.join(self.output_dir, d))]
        exp_num = 1
        if exps:
            nums = []
            for exp in exps:
                try: nums.append(int(exp.split("_")[1]))
                except ValueError: pass
            if nums: exp_num = max(nums) + 1
        self.experiment_dir = os.path.join(self.output_dir, f"experiment_{exp_num:03d}")
        self.model_dir = os.path.join(self.experiment_dir, "models")
        self.plot_dir = os.path.join(self.experiment_dir, "plots")
        self.log_dir = os.path.join(self.experiment_dir, "logs")
        self.pred_dir = os.path.join(self.experiment_dir, "predictions")
        os.makedirs(self.model_dir, exist_ok=True)
        os.makedirs(self.plot_dir, exist_ok=True)
        os.makedirs(self.log_dir, exist_ok=True)
        os.makedirs(self.pred_dir, exist_ok=True)
        with open(os.path.join(self.experiment_dir, "config.yaml"), "w") as f:
            yaml.dump(self.config, f, default_flow_style=False)

    def train_epoch(self) -> float:
        self.model.train()
        total_loss = 0.0
        self.optimizer.zero_grad()
        for step, (images, masks) in enumerate(self.train_loader):
            images, masks = images.to(self.device, non_blocking=True), masks.to(self.device, non_blocking=True)
            with autocast(enabled=(self.device.type == "cuda")):
                logits = self.model(images)
                loss = self.criterion(logits, masks) / self.accumulation_steps
            self.scaler.scale(loss).backward()
            if (step + 1) % self.accumulation_steps == 0 or (step + 1) == len(self.train_loader):
                if self.grad_clip_norm > 0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
            total_loss += loss.item() * self.accumulation_steps
        return total_loss / len(self.train_loader)

    def validate_epoch(self) -> Tuple[float, Dict[str, float]]:
        self.model.eval()
        total_loss = 0.0
        accumulated_metrics = { "precision": 0.0, "recall": 0.0, "f1_score": 0.0, "dice": 0.0, "iou": 0.0 }
        with torch.no_grad():
            for images, masks in self.val_loader:
                images, masks = images.to(self.device, non_blocking=True), masks.to(self.device, non_blocking=True)
                with autocast(enabled=(self.device.type == "cuda")):
                    logits = self.model(images)
                    loss = self.criterion(logits, masks)
                total_loss += loss.item()
                batch_metrics = compute_batch_metrics(logits, masks)
                for k, v in batch_metrics.items(): accumulated_metrics[k] += v
        num_steps = len(self.val_loader)
        avg_loss = total_loss / num_steps
        avg_metrics = {k: v / num_steps for k, v in accumulated_metrics.items()}
        return avg_loss, avg_metrics

    def fit(self):
        print(f"Starting training on {self.device}. Output directory: {self.experiment_dir}")
        for epoch in range(self.start_epoch, self.epochs + 1):
            epoch_start = time.time()
            train_loss = self.train_epoch()
            val_loss, val_metrics = self.validate_epoch()
            current_lr = self.optimizer.param_groups[0]["lr"]
            if self.scheduler is not None:
                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau): self.scheduler.step(val_loss)
                else: self.scheduler.step()
            epoch_duration = time.time() - epoch_start
            eta_seconds = epoch_duration * (self.epochs - epoch)
            eta_str = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))
            print(f"Epoch {epoch:02d}/{self.epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Dice: {val_metrics['dice']:.4f} | IoU: {val_metrics['iou']:.4f} | LR: {current_lr:.6f} | ETA: {eta_str}")
            self.history["epoch"].append(epoch)
            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["val_dice"].append(val_metrics["dice"])
            self.history["val_iou"].append(val_metrics["iou"])
            self.history["val_precision"].append(val_metrics["precision"])
            self.history["val_recall"].append(val_metrics["recall"])
            self.history["learning_rate"].append(current_lr)
            self._save_history_csv()
            self._save_qualitative_predictions(epoch)
            val_dice = val_metrics["dice"]
            is_best = val_dice > self.best_metric
            if is_best:
                self.best_metric = val_dice
                self.epochs_no_improve = 0
                self._save_checkpoint(epoch, is_best=True)
            else:
                self.epochs_no_improve += 1
            self._save_checkpoint(epoch, is_best=False)
            if self.epochs_no_improve >= self.early_stopping_patience:
                print(f"Early stopping triggered! No improvement in {self.early_stopping_patience} epochs.")
                break

    def _save_checkpoint(self, epoch: int, is_best: bool = False):
        checkpoint = { "epoch": epoch, "best_metric": self.best_metric, "model_state": self.model.state_dict(), "optimizer_state": self.optimizer.state_dict(), "scheduler_state": self.scheduler.state_dict() if self.scheduler else None, "scaler_state": self.scaler.state_dict() if self.scaler else None }
        torch.save(checkpoint, os.path.join(self.model_dir, "last_checkpoint.pth"))
        torch.save(self.optimizer.state_dict(), os.path.join(self.model_dir, "optimizer_state.pth"))
        if self.scheduler: torch.save(self.scheduler.state_dict(), os.path.join(self.model_dir, "scheduler_state.pth"))
        with open(os.path.join(self.model_dir, "config_used.yaml"), "w") as f:
            yaml.dump(self.config, f, default_flow_style=False)
        if is_best: torch.save(self.model.state_dict(), os.path.join(self.model_dir, "best_model.pth"))

    def load_checkpoint(self, checkpoint_path: str):
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state"])
        if self.scheduler and checkpoint.get("scheduler_state") is not None:
            self.scheduler.load_state_dict(checkpoint["scheduler_state"])
        if checkpoint.get("scaler_state") is not None:
            self.scaler.load_state_dict(checkpoint["scaler_state"])
        self.start_epoch = checkpoint["epoch"] + 1
        self.best_metric = checkpoint["best_metric"]
        history_file = os.path.join(self.log_dir, "training_history.csv")
        if os.path.exists(history_file):
            try:
                df = pd.read_csv(history_file)
                self.history = df.to_dict(orient="list")
            except Exception: pass

    def _save_history_csv(self):
        pd.DataFrame(self.history).to_csv(os.path.join(self.log_dir, "training_history.csv"), index=False)

    def _save_qualitative_predictions(self, epoch: int):
        self.model.eval()
        fig, axes = plt.subplots(self.num_val_samples, 3, figsize=(12, 4 * self.num_val_samples))
        fig.patch.set_facecolor("#0e1117")
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        with torch.no_grad():
            for idx in range(self.num_val_samples):
                image, mask = self.val_dataset[idx]
                image_dev = image.unsqueeze(0).to(self.device)
                with autocast(enabled=(self.device.type == "cuda")):
                    logits = self.model(image_dev)
                    probs = torch.sigmoid(logits)
                    pred = (probs > 0.5).float().squeeze(0).squeeze(0).cpu().numpy()
                img_np = image.permute(1, 2, 0).numpy()
                img_np = np.clip((img_np * std + mean) * 255.0, 0, 255).astype(np.uint8)
                mask_np = mask.squeeze(0).numpy()
                row_ax = axes[idx] if self.num_val_samples > 1 else axes
                row_ax[0].imshow(img_np)
                row_ax[0].set_title("Original Image", color="white", fontsize=10)
                row_ax[0].axis("off")
                row_ax[0].set_facecolor("#0e1117")
                row_ax[1].imshow(mask_np, cmap="gray")
                row_ax[1].set_title("Ground Truth Mask", color="white", fontsize=10)
                row_ax[1].axis("off")
                row_ax[1].set_facecolor("#0e1117")
                row_ax[2].imshow(pred, cmap="gray")
                row_ax[2].set_title(f"Prediction (Epoch {epoch})", color="white", fontsize=10)
                row_ax[2].axis("off")
                row_ax[2].set_facecolor("#0e1117")
        plt.tight_layout()
        plt.savefig(os.path.join(self.pred_dir, f"validation_grid_epoch_{epoch:03d}.png"), dpi=120, bbox_inches="tight", facecolor="#0e1117")
        plt.close()

In [ ]:
%%writefile src/segmentation/inference/inference.py
import os
import glob
from typing import Any, Optional, Tuple
import cv2
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

def visualize_overlay(image: np.ndarray, mask: np.ndarray, alpha: float = 0.4, color: Tuple[int, int, int] = (0, 255, 0)) -> np.ndarray:
    overlay = image.copy()
    mask_bool = mask > 127
    overlay[mask_bool] = color
    return cv2.addWeighted(image, 1.0 - alpha, overlay, alpha, 0)

def predict_image(model: nn.Module, image_path: str, device: torch.device, threshold: float = 0.5, transform: Optional[Any] = None) -> np.ndarray:
    model.eval()
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    if transform is not None:
        augmented = transform(image=image)
        input_tensor = augmented["image"]
    else:
        t = image.transpose(2, 0, 1) / 255.0
        t = torch.from_numpy(t).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        input_tensor = (t - mean) / std
    input_batch = input_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(input_batch)
        probs = torch.sigmoid(logits)
        prob_mask = probs.squeeze(0).squeeze(0).cpu().numpy()
    binary_mask = (prob_mask > threshold).astype(np.uint8) * 255
    return binary_mask

def predict_folder(model: nn.Module, input_dir: str, output_dir: str, device: torch.device, threshold: float = 0.5, transform: Optional[Any] = None, batch_size: int = 8, save_overlays: bool = False):
    os.makedirs(output_dir, exist_ok=True)
    if save_overlays:
        os.makedirs(os.path.join(output_dir, "overlays"), exist_ok=True)
    image_extensions = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(glob.glob(os.path.join(input_dir, f"*{ext}")))
        image_paths.extend(glob.glob(os.path.join(input_dir, f"*{ext.upper()}")))
    if not image_paths:
        print(f"No images found in {input_dir}")
        return
    model.eval()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        batch_images = []
        batch_tensors = []
        for path in batch_paths:
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            batch_images.append(img)
            if transform is not None:
                augmented = transform(image=img)
                batch_tensors.append(augmented["image"])
            else:
                t = img.transpose(2, 0, 1) / 255.0
                t = torch.from_numpy(t).float()
                t = (t - mean) / std
                batch_tensors.append(t)
        input_batch = torch.stack(batch_tensors).to(device)
        with torch.no_grad():
            logits = model(input_batch)
            probs = torch.sigmoid(logits)
            prob_masks = probs.squeeze(1).cpu().numpy()
        for idx, (img_path, prob_mask) in enumerate(zip(batch_paths, prob_masks)):
            binary_mask = (prob_mask > threshold).astype(np.uint8) * 255
            basename = os.path.basename(img_path)
            stem, _ = os.path.splitext(basename)
            mask_save_path = os.path.join(output_dir, f"{stem}_mask.png")
            cv2.imwrite(mask_save_path, binary_mask)
            if save_overlays:
                img_rgb = batch_images[idx]
                overlay = visualize_overlay(img_rgb, binary_mask)
                cv2.imwrite(os.path.join(output_dir, "overlays", f"{stem}_overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

## Step 2: Define Training Configuration

In [ ]:
%%writefile configs/train_config.yaml
project:
  name: "Route Resilience Model Training"
  phase: 2
  version: "1.0.0"
  random_seed: 42

model:
  architecture: "deeplabv3plus"
  encoder: "resnet34"
  encoder_weights: "imagenet"
  num_classes: 1

dataset:
  dataset_dir: "/kaggle/input/deepglobe-processed-dataset"
  image_size: 512
  occlusion_prob: 0.5
  num_workers: 2
  pin_memory: true
  persistent_workers: true

training:
  epochs: 15
  batch_size: 8
  learning_rate: 0.001
  weight_decay: 0.0001
  grad_clip_norm: 1.0
  early_stopping_patience: 5
  device: "cuda"
  accumulation_steps: 1

loss:
  bce_weight: 0.5
  dice_weight: 0.5
  jaccard_weight: 0.0

scheduler:
  type: "cosine"
  t_max: 15
  eta_min: 0.000001

paths:
  output_dir: "/kaggle/working/output"

## Step 3: Pipeline Orchestration & Training Execution

In [ ]:
import sys
import yaml
import random
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

# Ensure code modules are importable
sys.path.insert(0, os.path.abspath("."))

from src.segmentation.datasets.road_dataset import RoadDataset, SyntheticOcclusionTransform
from src.segmentation.models.model_factory import ModelFactory
from src.segmentation.losses.losses import CombinedLoss
from src.segmentation.training.trainer import Trainer

# 1. Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# 2. Load Config
with open("configs/train_config.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset_dir = config["dataset"]["dataset_dir"]
print("Training configuration successfully loaded.")

In [ ]:
# 3. Load Master Dataset and Splits
master_csv_path = os.path.join(dataset_dir, "master_dataset.csv")
if not os.path.exists(master_csv_path):
    raise FileNotFoundError(f"master_dataset.csv not found at {master_csv_path}.")

df = pd.read_csv(master_csv_path)
print(f"Master CSV contains {len(df)} tiles.")
print(df["split"].value_counts())

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
# Fallback if splits columns are missing/different
if len(val_df) == 0:
    print("⚠ Validation split empty. Creating 85/15 stratified random split based on unique source images...")
    unique_imgs = df["source_image_id"].unique()
    np.random.shuffle(unique_imgs)
    split_idx = int(len(unique_imgs) * 0.85)
    train_sources = set(unique_imgs[:split_idx])
    train_df = df[df["source_image_id"].isin(train_sources)].reset_index(drop=True)
    val_df = df[~df["source_image_id"].isin(train_sources)].reset_index(drop=True)
    print(f"New split created. Train: {len(train_df)} tiles, Val: {len(val_df)} tiles.")

# 4. Define Augmentations
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.2, rotate_limit=45, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 5. Datasets and Loaders
occ_transform = SyntheticOcclusionTransform(p=config["dataset"]["occlusion_prob"])

train_dataset = RoadDataset(train_df, dataset_dir, transform=train_transform, occlusion_transform=occ_transform)
val_dataset = RoadDataset(val_df, dataset_dir, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=True,
    num_workers=config["dataset"]["num_workers"],
    pin_memory=config["dataset"]["pin_memory"],
    persistent_workers=config["dataset"]["persistent_workers"]
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
    num_workers=config["dataset"]["num_workers"],
    pin_memory=config["dataset"]["pin_memory"],
    persistent_workers=config["dataset"]["persistent_workers"]
)

print(f"✓ DataLoaders successfully set up. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}.")

In [ ]:
# 6. Initialize Model, Loss, Optimizer and Scheduler
model = ModelFactory.create(
    config["model"]["architecture"],
    encoder_name=config["model"]["encoder"],
    encoder_weights=config["model"]["encoder_weights"],
    classes=config["model"]["num_classes"]
)

criterion = CombinedLoss(
    bce_weight=config["loss"]["bce_weight"],
    dice_weight=config["loss"]["dice_weight"],
    jaccard_weight=config["loss"]["jaccard_weight"]
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"]
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config["scheduler"]["t_max"],
    eta_min=config["scheduler"]["eta_min"]
)

print("✓ Model components successfully built.")

In [ ]:
# 7. Initialize Trainer and run training loop
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_dataset=val_dataset,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=config
)

# Run training
trainer.fit()

## Step 4: Plots and History Analytics

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.DataFrame(trainer.history)

# 1. Plot Loss curves
plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", color="#ff6b6b")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss", color="#4dabf7")
plt.title("Loss Curves", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.savefig(os.path.join(trainer.plot_dir, "loss_curve.png"), dpi=150)
plt.show()

# 2. Plot Metric curves
plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["val_dice"], label="Val Dice (F1)", color="#37b24d")
plt.plot(history_df["epoch"], history_df["val_iou"], label="Val IoU", color="#fcc419")
plt.title("Metric Curves", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.savefig(os.path.join(trainer.plot_dir, "metric_curve.png"), dpi=150)
plt.show()

print("✓ Analytics plots saved under:", trainer.plot_dir)

## Step 5: Exporting Best Predictions for Phase 3 Integration

In [ ]:
from src.segmentation.inference.inference import predict_image, visualize_overlay

# Load best trained model weights
best_weights_path = os.path.join(trainer.model_dir, "best_model.pth")
model.load_state_dict(torch.load(best_weights_path, map_location=trainer.device))
print("Loaded best model weights for final inference export.")

# Create target folder for Phase 3
phase3_preds_dir = os.path.join(trainer.output_dir, "predictions", "best_predictions")
os.makedirs(phase3_preds_dir, exist_ok=True)
print(f"Phase 3 input folder created: {phase3_preds_dir}")

# Export 10 sample validation images for direct consumption in Phase 3
sample_val_subset = val_df.head(10)
for idx, row in sample_val_subset.iterrows():
    img_path = resolve_tile_path(row["image_tile_path"], dataset_dir)
    basename = os.path.basename(img_path)
    stem, _ = os.path.splitext(basename)
    
    # Generate and save binary mask
    mask = predict_image(model, img_path, device=trainer.device, threshold=0.5, transform=val_transform)
    cv2.imwrite(os.path.join(phase3_preds_dir, f"{stem}_mask.png"), mask)
    
print(f"✓ Exported {len(sample_val_subset)} predictions to {phase3_preds_dir} ready for Phase 3 graph skeletonization.")